# 15b – Prophet-Prognose exportieren (für die Vergleichsgrafik)

**Nur nötig, falls Prophet NICHT in deinem `tft`-Kernel installiert ist.** In diesem Fall hier einmal in deinem **Prophet-Kernel** (der aus Notebook 01–12) *Run All* ausführen. Schreibt `../data/plot_prophet.csv`, die Notebook 15 dann einliest.

Muss dieselben Einstellungen nutzen wie Notebook 15: Station **Aotizhongxin**, Ursprung = Beginn Testjahr, 72 h, bestes Prophet-Modell (behandelt + Feiertag + Regressoren, Perfect Prognosis).

In [1]:
import numpy as np, pandas as pd
from pathlib import Path
from prophet import Prophet

STATION      = 'Aotizhongxin'
H            = 72
STARTOFFSET  = 0          # gleiche Zahl wie in Notebook 15
DATA_DIR     = Path('../data/prepared')
REGRESSOREN  = ['TEMP','DEWP','PRES','WSPM','RAIN','wd_sin','wd_cos']

def lade(v):
    tr = pd.read_csv(DATA_DIR/v/f'prophet_train_{STATION}.csv', parse_dates=['ds'])
    te = pd.read_csv(DATA_DIR/v/f'prophet_test_{STATION}.csv',  parse_dates=['ds'])
    return tr, te
def reg(df, cols):
    d = df.copy(); d['ds']=pd.to_datetime(d['ds']); d=d.set_index('ds').sort_index()
    d = d.reindex(pd.date_range(d.index.min(), d.index.max(), freq='h'))
    for c in cols: d[c]=d[c].interpolate(limit_direction='both')
    d.index.name='ds'; return d.reset_index()

tr, te = lade('behandelt')
ptrain = reg(tr, ['y']+REGRESSOREN)[['ds','y']+REGRESSOREN]
ptest  = reg(te, ['y']+REGRESSOREN)
fenster = ptest.iloc[STARTOFFSET:STARTOFFSET+H].copy()

m = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=True)
m.add_country_holidays(country_name='CN')
for r in REGRESSOREN: m.add_regressor(r)
m.fit(ptrain)
fc = m.predict(fenster[['ds']+REGRESSOREN])
out = pd.DataFrame({'ds': fenster['ds'].values, 'yhat': np.clip(fc['yhat'].values, 0, None)})
out.to_csv('../data/plot_prophet.csv', index=False)
print('gespeichert -> ../data/plot_prophet.csv'); print(out.head())

19:12:37 - cmdstanpy - INFO - Chain [1] start processing
19:12:50 - cmdstanpy - INFO - Chain [1] done processing


gespeichert -> ../data/plot_prophet.csv
                   ds        yhat
0 2016-03-01 00:00:00   71.689113
1 2016-03-01 01:00:00   85.589471
2 2016-03-01 02:00:00   94.213588
3 2016-03-01 03:00:00  106.087731
4 2016-03-01 04:00:00  105.862274
